# Fill Missing Report Table Statistics
This notebook loads report tables, detects empty variance/median/count values, fills them if possible, and records a final insight.

In [1]:
import pandas as pd
from pathlib import Path
from docx import Document

REPORT_DIR = Path(r'e:/tasnim/Graduation project model/tasnim/evaluation/statistical_analysis/reports')
FULL_REPORT_PATH = REPORT_DIR / 'evaluation_full_report.docx'
STAT_REPORT_PATH = REPORT_DIR / 'statistical_analysis_report.docx'
print('Report directory:', REPORT_DIR)
print('Full report exists:', FULL_REPORT_PATH.exists())
print('Statistical report exists:', STAT_REPORT_PATH.exists())

Report directory: e:\tasnim\Graduation project model\tasnim\evaluation\statistical_analysis\reports
Full report exists: True
Statistical report exists: True


## Load report tables into DataFrames
Read the Word report tables and inspect their structure and missing values.

In [2]:
def docx_tables_to_dataframes(doc_path: Path):
    doc = Document(str(doc_path))
    tables = []
    for table in doc.tables:
        headers = [cell.text.strip() for cell in table.rows[0].cells]
        rows = []
        for row in table.rows[1:]:
            rows.append({headers[i]: row.cells[i].text.strip() for i in range(len(headers))})
        tables.append((headers, pd.DataFrame(rows)))
    return tables

full_tables = docx_tables_to_dataframes(FULL_REPORT_PATH)
stat_tables = docx_tables_to_dataframes(STAT_REPORT_PATH)
print('Full report table count:', len(full_tables))
print('Stat report table count:', len(stat_tables))
for i, (headers, df) in enumerate(stat_tables, 1):
    print(f'\nStat report table {i}')
    print('Headers:', headers)
    print(df.head(3).to_dict(orient='records'))

IndexError: tuple index out of range

In [3]:
# Debug table structure and row lengths for STAT_REPORT_PATH
doc = Document(str(STAT_REPORT_PATH))
for ti, table in enumerate(doc.tables, 1):
    print('Table', ti, 'rows', len(table.rows), 'cols', len(table.columns))
    for ri, row in enumerate(table.rows):
        print(' row', ri, 'cells', len(row.cells), 'texts', [cell.text for cell in row.cells])
    print('---')


Table 1 rows 7 cols 16
 row 0 cells 16 texts ['Metric', 'Metric', 'Mean', 'Std Dev', 'Std Dev', 'Std Dev', 'Variance', 'Variance', 'Min', 'Min', 'Max', 'Median', 'Median', 'Count', 'Count', 'Count']
 row 1 cells 15 texts ['accuracy', '0.882541701055032', '0.882541701055032', '0.882541701055032', '0.01936455760358606', '', '', '0.8484679644651406', '0.8484679644651406', '0.915820638940861', '0.915820638940861', '0.915820638940861', '', '', '']
 row 2 cells 15 texts ['precision', '0.8720270798086571', '0.8720270798086571', '0.8720270798086571', '0.025589078274550536', '', '', '0.8275905773618031', '0.8275905773618031', '0.9286985084870074', '0.9286985084870074', '0.9286985084870074', '', '', '']
 row 3 cells 15 texts ['recall', '0.8838412174386048', '0.8838412174386048', '0.8838412174386048', '0.02557295703021548', '', '', '0.8375596833560659', '0.8375596833560659', '0.9295438301140682', '0.9295438301140682', '0.9295438301140682', '', '', '']
 row 4 cells 15 texts ['f1_score', '0.8750673

In [4]:
# Print only summary of structure and first few rows
for ti, table in enumerate(doc.tables, 1):
    print('Table', ti, 'rows', len(table.rows), 'cols', len(table.columns))
    for ri, row in enumerate(table.rows[:5]):
        print(' row', ri, 'cells', len(row.cells), [cell.text for cell in row.cells])
    if len(table.rows) > 5:
        print(' ...', len(table.rows)-5, 'more rows')
    print('---')


Table 1 rows 7 cols 16
 row 0 cells 16 ['Metric', 'Metric', 'Mean', 'Std Dev', 'Std Dev', 'Std Dev', 'Variance', 'Variance', 'Min', 'Min', 'Max', 'Median', 'Median', 'Count', 'Count', 'Count']
 row 1 cells 15 ['accuracy', '0.882541701055032', '0.882541701055032', '0.882541701055032', '0.01936455760358606', '', '', '0.8484679644651406', '0.8484679644651406', '0.915820638940861', '0.915820638940861', '0.915820638940861', '', '', '']
 row 2 cells 15 ['precision', '0.8720270798086571', '0.8720270798086571', '0.8720270798086571', '0.025589078274550536', '', '', '0.8275905773618031', '0.8275905773618031', '0.9286985084870074', '0.9286985084870074', '0.9286985084870074', '', '', '']
 row 3 cells 15 ['recall', '0.8838412174386048', '0.8838412174386048', '0.8838412174386048', '0.02557295703021548', '', '', '0.8375596833560659', '0.8375596833560659', '0.9295438301140682', '0.9295438301140682', '0.9295438301140682', '', '', '']
 row 4 cells 15 ['f1_score', '0.8750673747926584', '0.875067374792658

In [5]:
# Summarize STAT_REPORT_PATH table dimensions and missing-stat columns only
for ti, table in enumerate(doc.tables, 1):
    header_text = [cell.text for cell in table.rows[0].cells]
    row_counts = len(table.rows)
    col_counts = len(table.columns)
    print('Table', ti, 'rows', row_counts, 'cols', col_counts, 'headers', header_text)
    for col in ['Variance', 'Median', 'Count', 'variance', 'median', 'count']:
        if col in header_text:
            missing = sum(1 for row in table.rows[1:] if (row.cells[header_text.index(col)].text.strip() == ''))
            print('  ', col, 'missing', missing)
    print('---')


Table 1 rows 7 cols 16 headers ['Metric', 'Metric', 'Mean', 'Std Dev', 'Std Dev', 'Std Dev', 'Variance', 'Variance', 'Min', 'Min', 'Max', 'Median', 'Median', 'Count', 'Count', 'Count']
   Variance missing 6
   Median missing 0
   Count missing 6
---
Table 2 rows 7 cols 4 headers ['Metric', 'Mean', 'CI Lower', 'CI Upper']
---
Table 3 rows 7 cols 7 headers ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median']
   Variance missing 6
   Median missing 6
---
Table 4 rows 7 cols 7 headers ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median']
   Variance missing 6
   Median missing 6
---
Table 5 rows 7 cols 7 headers ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median']
   Variance missing 6
   Median missing 6
---
Table 6 rows 7 cols 7 headers ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median']
   Variance missing 6
   Median missing 6
---
Table 7 rows 7 cols 7 headers ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median']
   Varian

In [6]:
# Show the first 3 rows and column text for table 1 only
for j, row in enumerate(doc.tables[0].rows[:4]):
    print('Row', j, [cell.text for cell in row.cells])
    print('Lengths:', len(row.cells))


Row 0 ['Metric', 'Metric', 'Mean', 'Std Dev', 'Std Dev', 'Std Dev', 'Variance', 'Variance', 'Min', 'Min', 'Max', 'Median', 'Median', 'Count', 'Count', 'Count']
Lengths: 16
Row 1 ['accuracy', '0.882541701055032', '0.882541701055032', '0.882541701055032', '0.01936455760358606', '', '', '0.8484679644651406', '0.8484679644651406', '0.915820638940861', '0.915820638940861', '0.915820638940861', '', '', '']
Lengths: 15
Row 2 ['precision', '0.8720270798086571', '0.8720270798086571', '0.8720270798086571', '0.025589078274550536', '', '', '0.8275905773618031', '0.8275905773618031', '0.9286985084870074', '0.9286985084870074', '0.9286985084870074', '', '', '']
Lengths: 15
Row 3 ['recall', '0.8838412174386048', '0.8838412174386048', '0.8838412174386048', '0.02557295703021548', '', '', '0.8375596833560659', '0.8375596833560659', '0.9295438301140682', '0.9295438301140682', '0.9295438301140682', '', '', '']
Lengths: 15


In [ ]:
SUMMARY_DIR = REPORT_DIR.parent / 'summary'
print('Summary dir:', SUMMARY_DIR.exists())
for path in SUMMARY_DIR.glob('*.json'):
    print('Found', path.name)

global_stats = pd.read_json(SUMMARY_DIR / 'global_metrics.json') if (SUMMARY_DIR / 'global_metrics.json').exists() else None
per_batch = pd.read_json(SUMMARY_DIR / 'per_batch_metrics.json') if (SUMMARY_DIR / 'per_batch_metrics.json').exists() else None
print('Global stats shape:', None if global_stats is None else global_stats.shape)
print('Per batch exists:', per_batch is not None)
if global_stats is not None:
    display(global_stats.head())


In [ ]:
print('Summary path exists:', (SUMMARY_DIR / 'global_metrics.json').exists())
print('Summary path exists:', (SUMMARY_DIR / 'per_batch_metrics.json').exists())


## Inspect missing statistic columns
Identify which tables have empty or missing variance/median/count fields.

In [ ]:
def missing_statistics(df: pd.DataFrame):
    missing = {}
    for col in ['Variance','Median','Count','variance','median','count']:
        if col in df.columns:
            mask = df[col].astype(str).str.strip() == ''
            missing[col] = mask.sum()
    return missing

for i, (headers, df) in enumerate(stat_tables, 1):
    miss = missing_statistics(df)
    print(f'\nTable {i} missing summary: {miss}')
    print(df.head(3))

## Final Insight
Summarize what we found in the report tables and whether the missing summary statistics can be filled.